In [ ]:
def fetch_sample_puuids(tier="GOLD", division="I", n=5):
    url = f"https://na1.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{division}"
    
    res = requests.get(url, headers=HEADERS)
    
    if res.status_code != 200:
        print("❌ error:", res.status_code)
        return []
    
    data = res.json()
    
    puuids = []
    for entry in data[:n]:
        if "puuid" in entry:
            puuids.append(entry["puuid"])
    
    print(f"✅ got {len(puuids)} puuids")
    return puuids

In [ ]:
def fetch_multi_tier_puuids():
    tiers = ["SILVER", "GOLD", "PLATINUM"]
    puuids = []

    for t in tiers:
        puuids += fetch_sample_puuids(tier=t, n=3)

    return puuids

In [ ]:
import requests
import time
import datetime

CUTOFF = int(datetime.datetime(2025, 10, 22).timestamp() * 1000)


def get_match_ids(puuid, count=100):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    params = {"start": 0, "count": count}
    return requests.get(url, headers=HEADERS, params=params).json()


def get_match_end_time(match_id):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    res = requests.get(url, headers=HEADERS)

    if res.status_code != 200:
        return None
    
    data = res.json()
    return data["info"].get("gameEndTimestamp")


def test_time_filter(puuid_list, count_per_user=100):
    total = 0
    valid = 0

    for puuid in puuid_list:
        match_ids = get_match_ids(puuid, count_per_user)

        if not match_ids:
            continue

        for mid in match_ids:
            t = get_match_end_time(mid)
            if t is None:
                continue

            total += 1

            if t < CUTOFF:
                valid += 1

            time.sleep(0.1)  # 防429

    print("\n====== RESULT ======")
    print("Total matches:   ", total)
    print("Before Oct 22:   ", valid)
    print("Filtered out:    ", total - valid)
    print("Keep ratio:      ", round(valid / total, 3) if total else 0)

In [ ]:
puuids = fetch_multi_tier_puuids()
test_time_filter(puuids)